# Purpose:
# Authenticate Databricks Spark to ADLS Gen2 using service principal credentials from Databricks secret scope.

# Define secret scope and storage account variables
# This notebook configures Spark OAuth access to ADLS Gen2.
# Secrets are read from the Databricks secret scope. No secret values are hardcoded.

In [ ]:
# Define secret scope and key names
secret_scope = "kv-oncom-dev-scope"
client_secret_key = "client-secret"
app_id_key = "app-id"
tenant_id_key = "tenant-id"

# Define ADLS Gen2 storage account name
storage_account = "stoncomdev001"

# Read service principal secrets
# Read the client secret, application ID, and tenant ID from Databricks secrets.

In [ ]:
# Fetch credentials from Databricks secret scope
service_credential = dbutils.secrets.get(scope=secret_scope, key=client_secret_key)
appid = dbutils.secrets.get(scope=secret_scope, key=app_id_key)
tenantid = dbutils.secrets.get(scope=secret_scope, key=tenant_id_key)

# Validate tenant ID format
# The tenant ID must be only the tenant GUID.
# It must not contain a URL, microsoftonline, /oauth2/token, or slashes.

In [ ]:
# Print validation checks for the tenant ID format
print("tenantid length:", len(tenantid))
print("tenantid starts with https:", tenantid.startswith("https"))
print("tenantid contains microsoftonline:", "microsoftonline" in tenantid)
print("tenantid contains slash:", "/" in tenantid)

# Configure Spark OAuth access to ADLS Gen2
# Configure Spark to use service principal OAuth authentication for the ADLS Gen2 storage account.

In [ ]:
# Set Spark configuration for ADLS Gen2 OAuth authentication
spark.conf.set(
    f"fs.azure.account.auth.type.{storage_account}.dfs.core.windows.net",
    "OAuth"
)

# Set the OAuth token provider class
spark.conf.set(
    f"fs.azure.account.oauth.provider.type.{storage_account}.dfs.core.windows.net",
    "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider"
)

# Pass the client ID to the configuration
spark.conf.set(
    f"fs.azure.account.oauth2.client.id.{storage_account}.dfs.core.windows.net",
    appid
)

# Pass the client secret to the configuration
spark.conf.set(
    f"fs.azure.account.oauth2.client.secret.{storage_account}.dfs.core.windows.net",
    service_credential
)

# Construct and set the Azure AD token endpoint URL
spark.conf.set(
    f"fs.azure.account.oauth2.client.endpoint.{storage_account}.dfs.core.windows.net",
    f"https://microsoftonline.com{tenantid}/oauth2/token"
)

print("ADLS OAuth configuration completed.")

# Test ADLS container access
# List the ADLS container root to confirm authentication and storage access are working.

In [ ]:
# Test connectivity by listing the root directory of the container
display(dbutils.fs.ls("abfss://oaonoperationsdev@stoncomdev001.dfs.core.windows.net/"))